
# Mokhles Group HR Analytics — Bangladesh

**Exploratory analysis of a synthetic FY2025 HR portfolio dataset**

This notebook analyses the consolidated Mokhles Group master workbook and demonstrates practical HR analytics for:

- workforce composition and headcount
- hiring and employee exits
- gender representation
- compensation and payroll
- performance distribution
- health and safety indicators

> **Data note:** All employee names, IDs, salaries, performance outcomes and incidents are fictional and generated for learning and portfolio purposes.


In [ ]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


## 1. Locate the master workbook

In [ ]:

def find_master_workbook() -> Path:
    search_roots = [Path("/kaggle/input"), Path.cwd(), Path.cwd().parent]
    patterns = [
        "**/Mokhles_Group_HR_Analytics_Master_Dataset_FY2025.xlsx",
        "**/*Master*HR*Dataset*FY2025*.xlsx",
    ]
    for root in search_roots:
        if not root.exists():
            continue
        for pattern in patterns:
            matches = sorted(root.glob(pattern))
            if matches:
                return matches[0]
    raise FileNotFoundError(
        "Master workbook not found. Add the Mokhles Group HR dataset as a Kaggle input."
    )

MASTER_FILE = find_master_workbook()
print(f"Master workbook: {MASTER_FILE}")


## 2. Load the analytical sheets

In [ ]:

# The first three rows contain report titles and subtitles; row 4 contains headers.
SHEETS = {
    "employees": "Employee Master",
    "monthly": "Monthly HR KPI",
    "recruitment": "Recruitment Master",
    "separations": "Employee Separations",
    "leave": "Leave Master",
    "training": "Training Master",
    "compensation": "Compensation Master",
    "performance": "Performance Master",
    "hse": "HSE Master",
}

data = {
    key: pd.read_excel(MASTER_FILE, sheet_name=sheet_name, header=3)
    for key, sheet_name in SHEETS.items()
}

for key, frame in data.items():
    frame.dropna(axis=1, how="all", inplace=True)
    print(f"{key:14s}: {frame.shape[0]:,} rows × {frame.shape[1]} columns")


## 3. Executive HR KPI summary

In [ ]:

employees = data["employees"]
monthly = data["monthly"]
separations = data["separations"]
recruitment = data["recruitment"]
training = data["training"]
hse = data["hse"]

kpis = pd.Series({
    "Year-end headcount": len(employees),
    "FY2025 hires": int(monthly["Hires"].sum()),
    "FY2025 exits": len(separations),
    "Annual turnover rate": separations.shape[0] / monthly["Average Headcount"].mean(),
    "Recruitment requisitions": len(recruitment),
    "Training participations": len(training),
    "HSE records": len(hse),
    "Annual payroll cost (BDT)": monthly["Payroll Cost (BDT)"].sum(),
}, name="Value")

display(kpis.to_frame())


## 4. Workforce composition

In [ ]:

department_headcount = employees["Department"].value_counts().sort_values()

ax = department_headcount.plot(kind="barh", figsize=(10, 6), title="Year-End Headcount by Department")
ax.set_xlabel("Employees")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


In [ ]:

gender_profile = employees["Gender"].value_counts(dropna=False)
display(pd.DataFrame({
    "Headcount": gender_profile,
    "Representation": gender_profile / gender_profile.sum()
}))

ax = gender_profile.plot(kind="bar", figsize=(7, 4), title="Gender Representation")
ax.set_xlabel("")
ax.set_ylabel("Employees")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 5. Monthly workforce movement

In [ ]:

monthly = monthly.copy()
monthly["Month"] = pd.to_datetime(monthly["Month"])
monthly = monthly.sort_values("Month")

ax = monthly.set_index("Month")[["Hires", "Exits"]].plot(
    figsize=(10, 5), marker="o", title="Monthly Hires and Exits"
)
ax.set_ylabel("Employees")
ax.set_xlabel("")
plt.tight_layout()
plt.show()


## 6. Turnover and separation reasons

In [ ]:

exit_reasons = separations["Exit Reason"].value_counts().sort_values()
display(exit_reasons.to_frame("Employees"))

ax = exit_reasons.plot(kind="barh", figsize=(9, 5), title="Primary Reasons for Employee Exit")
ax.set_xlabel("Employees")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 7. Compensation and payroll analysis

In [ ]:

compensation = data["compensation"]
payroll_by_department = (
    compensation.groupby("Department")["Monthly Gross"]
    .agg(["count", "sum", "mean", "median"])
    .rename(columns={"count": "Headcount", "sum": "Monthly Payroll", "mean": "Average Gross", "median": "Median Gross"})
    .sort_values("Monthly Payroll", ascending=False)
)
display(payroll_by_department)

ax = payroll_by_department["Monthly Payroll"].sort_values().plot(
    kind="barh", figsize=(10, 6), title="Monthly Gross Payroll by Department"
)
ax.set_xlabel("BDT")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 8. Performance distribution

In [ ]:

performance = data["performance"]
rating_order = [
    "Needs Improvement",
    "Partially Meets",
    "Meets Expectations",
    "Exceeds Expectations",
    "Outstanding",
]
rating_counts = performance["Rating Label"].value_counts().reindex(rating_order, fill_value=0)
display(rating_counts.to_frame("Employees"))

ax = rating_counts.plot(kind="bar", figsize=(10, 4), title="Performance Rating Distribution")
ax.set_xlabel("")
ax.set_ylabel("Employees")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## 9. Health and safety profile

In [ ]:

incident_profile = hse["Incident Type"].value_counts().sort_values()
display(incident_profile.to_frame("Records"))

ax = incident_profile.plot(kind="barh", figsize=(9, 5), title="Health and Safety Records by Incident Type")
ax.set_xlabel("Records")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 10. Data quality checks

In [ ]:

quality_summary = []
for name, frame in data.items():
    quality_summary.append({
        "Dataset": name,
        "Rows": len(frame),
        "Columns": frame.shape[1],
        "Duplicate Rows": int(frame.duplicated().sum()),
        "Missing Cells": int(frame.isna().sum().sum()),
        "Missing Cell Rate": frame.isna().sum().sum() / max(1, frame.size),
    })

quality_summary = pd.DataFrame(quality_summary).sort_values("Rows", ascending=False)
display(quality_summary)



## Conclusion

The Mokhles Group HR portfolio demonstrates how a connected HR reporting ecosystem can support:

- workforce planning and organisational design
- recruitment-funnel and hiring-efficiency analysis
- turnover and retention diagnosis
- compensation and performance review
- diversity, learning and safety governance

The dataset is synthetic, making it suitable for public portfolio work without exposing real employee information.
